In [1]:
pip install selenium pandas webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [ ]:
SCRAPPING FRANCE

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
import pandas as pd
import time
import re
import os

# ==================== CONFIGURATION ====================
BASE_SEARCH_URL = "https://www.helloasso.com/e/recherche?query=scouts-et-guides-de-france"

options = webdriver.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--start-maximized')
options.add_argument('--disable-notifications')
options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
wait = WebDriverWait(driver, 25)

# ==================== FONCTIONS ====================

def get_all_association_links_with_pagination():
    """
    Version CORRIGÉE : parcourt TOUTES les pages de pagination
    pour récupérer TOUTES les associations (pas seulement la 1ère page)
    """
    print(f"🔍 Début de la collecte des liens d'associations...")
    driver.get(BASE_SEARCH_URL)
    time.sleep(4)

    all_links = []
    
    # 1. Cliquer sur "Tout voir"
    try:
        tout_voir_button = driver.find_element(
            By.CSS_SELECTOR, 
            "button[data-testid='Explore_LP_Carousel_ShowAllResults_Link']"
        )
        driver.execute_script("arguments[0].click();", tout_voir_button)
        print("✅ 'Tout voir' cliqué.")
        time.sleep(5)  # Attendre le chargement de la nouvelle page
    except Exception as e:
        print(f"❌ Impossible de cliquer sur 'Tout voir' : {e}")
        return []
    
    # 2. MAIN TENANT on est sur la page avec PAGINATION (1, 2, 3...)
    # On va parcourir TOUTES les pages
    
    page_num = 1
    max_pages = 9  
    
    while page_num <= max_pages:
        print(f"\n📄 PAGE {page_num} - Extraction des associations...")
        
        # Attendre que les cartes d'association soient chargées
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/associations/']")))
        except:
            print("   ⏳ Attente du chargement des associations...")
            time.sleep(3)
        
        # Extraire TOUTES les URLs d'associations de cette page
        association_links_on_page = []
        
        # Chercher TOUS les liens vers des associations
        association_elements = driver.find_elements(By.CSS_SELECTOR, "a[href*='/associations/']")
        
        for elem in association_elements:
            try:
                href = elem.get_attribute('href')
                if href and '/associations/' in href:
                    # Filtrer pour n'avoir que les pages PRINCIPALES d'associations
                    if not any(subpage in href for subpage in ['/adhesions/', '/evenements/', '/formulaires/', '/collectes/']):
                        if href not in all_links:
                            association_links_on_page.append(href)
            except StaleElementReferenceException:
                continue
        
        # Ajouter les nouveaux liens à la liste globale
        new_count = len(association_links_on_page)
        all_links.extend(association_links_on_page)
        
        print(f"   ✅ {new_count} nouvelles associations trouvées sur cette page")
        print(f"   📊 Total cumulé : {len(all_links)} associations")
        
        # 3. Trouver et cliquer sur "Suivant" ou le prochain numéro de page
        try:
            # D'abord essayer de trouver le bouton "Suivant" avec data-ux
            next_button = driver.find_element(By.CSS_SELECTOR, "button[data-ux='Explore_Search_Pagination_next']")
            
            # Vérifier si le bouton est désactivé (dernière page)
            if "disabled" in next_button.get_attribute("class") or next_button.get_attribute("aria-disabled") == "true":
                print("✅ Dernière page atteinte (bouton Suivant désactivé).")
                break
            
            print("➡️  Passage à la page suivante...")
            driver.execute_script("arguments[0].click();", next_button)
            page_num += 1
            time.sleep(4)  
        except NoSuchElementException:
            # Si pas de bouton "Suivant", chercher les numéros de page
            try:
                # Chercher tous les boutons de pagination (1, 2, 3...)
                page_buttons = driver.find_elements(By.CSS_SELECTOR, "button.Pagination_plus_1, button.Pagination_plus_2, button.Pagination_plus_3")
                next_page_found = False
                for btn in page_buttons:
                    try:
                        btn_text = btn.text.strip()
                        if btn_text.isdigit() and int(btn_text) == page_num + 1:
                            print(f"📄 Passage à la page {btn_text}...")
                            driver.execute_script("arguments[0].click();", btn)
                            next_page_found = True
                            page_num += 1
                            time.sleep(4)
                            break
                    except:
                        continue
                
                if not next_page_found:
                    print("✅ Plus de pages suivantes disponibles.")
                    break
                    
            except Exception as e:
                print(f"❌ Impossible de trouver la pagination : {e}")
                break
    
    print(f"\n🎯 COLLECTE TERMINÉE : {len(all_links)} associations uniques trouvées sur {page_num} pages.")
    return all_links

def scrape_events_from_association(association_url):
    """VOTRE CODE EXISTANT pour scraper les événements d'une association"""
    print(f"\n   🏕️  Scraping de : {association_url}")
    
    # Ouvrir la page de l'association
    driver.get(association_url)
    time.sleep(3)
    
    events_data = []
    
    print("🌐 Page chargée, recherche des événements scouts...")

    # -----------------------------
    # 1️⃣ CLIQUEZ SUR "VOIR TOUS LES ÉVÉNEMENTS"
    # -----------------------------
    try:
        print("🔍 Recherche du bouton 'Voir tous les événements'...")
        
        button_selectors = [
            "//button[contains(., 'Voir tous les événements')]",
            "//button[contains(@data-ux, 'ShowAllActions')]",
            "//button[contains(@class, 'HaButton') and contains(., 'Voir tous')]",
            "//button[.//span[contains(., 'Voir tous les événements')]]"
        ]
        
        voir_tous_button = None
        for selector in button_selectors:
            try:
                voir_tous_button = driver.find_element(By.XPATH, selector)
                if voir_tous_button and voir_tous_button.is_displayed():
                    print(f"✅ Bouton trouvé avec sélecteur: {selector}")
                    break
            except:
                continue
        
        if voir_tous_button:
            driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", voir_tous_button)
            time.sleep(2)
            driver.execute_script("arguments[0].click();", voir_tous_button)
            print("✅ Bouton 'Voir tous les événements' cliqué !")
            time.sleep(4)
        else:
            print("⚠️ Bouton non trouvé, continuer sans cliquer...")
            
    except Exception as e:
        print(f"❌ Erreur avec le bouton: {e}")
        print("ℹ️ Continuation sans cliquer sur le bouton...")

    # -----------------------------
    # 2️⃣ ATTENDRE LE CHARGEMENT DES ÉVÉNEMENTS
    # -----------------------------
    print("⏳ Attente du chargement des événements...")
    time.sleep(4)

    # Faire défiler pour charger tous les événements
    print("📜 Défilement pour charger tous les événements...")
    for i in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        print(f"  Défilement {i+1}/3 terminé")

    time.sleep(3)

    # -----------------------------
    # 3️⃣ TROUVER TOUS LES CONTENEURS D'ÉVÉNEMENTS
    # -----------------------------
    print("\n🔍 Recherche des conteneurs d'événements...")
    
    event_containers = []
    container_selectors = [
        "//div[contains(@class, 'ActionContent--Text')]",
        "//div[contains(@class, 'ActionCard')]",
        "//div[contains(@class, 'campaign-card')]",
        "//article[contains(@class, 'card')]",
        "//div[.//h3[contains(., 'Camp') or contains(., 'BAFA') or contains(., 'Participation') or contains(., 'WIG')]]"
    ]
    
    for selector in container_selectors:
        try:
            containers = driver.find_elements(By.XPATH, selector)
            if containers:
                print(f"✅ {len(containers)} conteneurs trouvés avec: {selector}")
                event_containers.extend(containers)
        except:
            continue

    # Supprimer les doublons
    unique_containers = []
    seen_elements = set()
    for container in event_containers:
        if container.id not in seen_elements:
            seen_elements.add(container.id)
            unique_containers.append(container)

    print(f"📊 Total de conteneurs uniques: {len(unique_containers)}")

    # Recherche alternative si peu de conteneurs
    if len(unique_containers) < 2:
        print("🔄 Recherche alternative par titres h3...")
        h3_elements = driver.find_elements(By.XPATH, "//h3")
        for h3 in h3_elements:
            text = h3.text.strip()
            if text and any(keyword in text.lower() for keyword in ['camp', 'bafa', 'participation', 'wig', 'farfa', 'scout']):
                try:
                    container = h3.find_element(By.XPATH, "./ancestor::div[contains(@class, 'card') or contains(@class, 'Action')]")
                    if container and container.id not in seen_elements:
                        unique_containers.append(container)
                        seen_elements.add(container.id)
                except:
                    pass

    # -----------------------------
    # 4️⃣ EXTRAIRE LES DONNÉES DE CHAQUE ÉVÉNEMENT
    # -----------------------------
    if len(unique_containers) == 0:
        print("ℹ️ Aucun événement trouvé dans cette association.")
        return events_data
    
    print(f"\n🎯 Extraction des données pour {len(unique_containers)} événements...")

    for i, container in enumerate(unique_containers):
        try:
            # Titre
            title = "Titre non trouvé"
            try:
                title_element = container.find_element(By.XPATH, ".//h3")
                title = title_element.text.strip()
            except:
                try:
                    all_text = container.text.strip()
                    lines = all_text.split('\n')
                    for line in lines:
                        if line and len(line) > 10 and any(keyword in line.lower() for keyword in ['camp', 'bafa', 'participation', 'wig']):
                            title = line.strip()
                            break
                except:
                    pass
            
            # Prix
            price = "Prix non trouvé"
            try:
                price_element = container.find_element(By.XPATH, ".//p[contains(@class, 'Number') and contains(@class, 'BasePrice')]")
                price_text = price_element.text.strip()
                price_match = re.search(r'(\d+\s*€)', price_text)
                if price_match:
                    price = price_match.group(1)
                else:
                    price = price_text
            except:
                try:
                    container_html = container.get_attribute('innerHTML')
                    price_matches = re.findall(r'>\s*(\d+\s*€)\s*<', container_html)
                    if price_matches:
                        price = price_matches[0]
                except:
                    pass
            
            # Description
            description = "Description non trouvée"
            try:
                description_element = container.find_element(By.XPATH, ".//h3/following-sibling::p[1]")
                description = description_element.text.strip()
                if len(description) > 150:
                    description = description[:150] + "..."
            except:
                pass
            
            # Date
            event_date = "Date non spécifiée"
            try:
                date_patterns = [
                    r'(\d{1,2}[/-]\d{1,2}[/-]\d{4})',
                    r'(\d{1,2}\s+[a-zA-Zéèêëàâäôöûüç]+\s+\d{4})',
                    r'(\d{1,2}\s*[-–]\s*\d{1,2}\s+[a-zA-Zéèêëàâäôöûüç]+\s+\d{4})',
                    r'(\d{1,2}\s*[-–]\s*\d{1,2}\s*[/-]\s*\d{1,2}\s*[/-]\s*\d{4})'
                ]
                
                for pattern in date_patterns:
                    date_match = re.search(pattern, title, re.IGNORECASE)
                    if date_match:
                        event_date = date_match.group(1)
                        break
                
                if event_date == "Date non spécifiée":
                    container_text = container.text
                    for pattern in date_patterns:
                        date_match = re.search(pattern, container_text, re.IGNORECASE)
                        if date_match:
                            event_date = date_match.group(1)
                            break
            except:
                pass
            
            # Type d'événement
            event_type = "Activité"
            title_lower = title.lower()
            
            if 'camp' in title_lower:
                event_type = "Camp"
            elif 'bafa' in title_lower:
                event_type = "Formation BAFA"
            elif 'wig' in title_lower:
                event_type = "Week-end d'intégration"
            elif 'farfa' in title_lower:
                event_type = "Farfadets"
            elif 'louveteau' in title_lower:
                event_type = "Louveteaux"
            elif 'scout' in title_lower and 'guide' not in title_lower:
                event_type = "Scouts"
            elif 'pionnier' in title_lower:
                event_type = "Pionniers"
            elif 'tenue' in title_lower or 'commande' in title_lower:
                event_type = "Commande matériel"
            
            # Organisation
            org_name = association_url.split('/')[-1].replace('-', ' ').title()
            
            # Lieu
            location = org_name.split()[-1] if len(org_name.split()) > 1 else "France"
            
            if 'château' in description.lower() or 'jambville' in description.lower():
                location = "Château de Jambville, France"
            elif 'territoire' in description.lower():
                location = f"Territoire de {org_name.split()[-1]}, France"
            
            # Ajouter aux données
            events_data.append({
                "Association": org_name,
                "URL_Association": association_url,
                "Titre": title,
                "Type": event_type,
                "Date": event_date,
                "Prix": price,
                "Lieu": location,
                "Description": description
            })
            
            print(f"  ✅ Événement {i+1}: {title[:50]}...")
            
        except Exception as e:
            print(f"  ❌ Erreur sur l'événement {i+1}: {str(e)[:50]}")
            continue
    
    return events_data

# ==================== PROGRAMME PRINCIPAL ====================

print("🚀 DÉBUT DU SCRAPING COMPLET DES SCOUTS ET GUIDES DE FRANCE")
print("=" * 60)

# Étape 1: Récupérer TOUTES les URLs d'associations (toutes les pages)
all_association_links = get_all_association_links_with_pagination()

if not all_association_links:
    print("❌ Aucune association trouvée. Arrêt du script.")
    driver.quit()
    exit()

# Sauvegarder temporairement la liste des URLs
urls_filename = f"liste_associations_scouts_{time.strftime('%Y%m%d_%H%M%S')}.txt"
with open(urls_filename, 'w', encoding='utf-8') as f:
    for url in all_association_links:
        f.write(url + '\n')
print(f"📄 Liste des URLs sauvegardée dans: {urls_filename}")

# Étape 2: Pour chaque association, scraper les événements
all_events = []
association_count = 0

# LIMITE pour les tests
MAX_ASSOCIATIONS = len(all_association_links) # Scrape 10 associations max pour tester
if len(all_association_links) > MAX_ASSOCIATIONS:
    print(f"\n⚠️  MODE TEST: Scraping de {MAX_ASSOCIATIONS} associations sur {len(all_association_links)}")
    all_association_links = all_association_links[:MAX_ASSOCIATIONS]

for i, assoc_url in enumerate(all_association_links):
    association_count += 1
    print(f"\n{'='*60}")
    print(f"ASSOCIATION {i+1}/{len(all_association_links)}")
    
    try:
        events = scrape_events_from_association(assoc_url)
        if events:
            all_events.extend(events)
            print(f"   ✅ {len(events)} événements ajoutés")
        else:
            print("   ℹ️  Aucun événement trouvé dans cette association")
            
        # Pause pour ne pas surcharger le serveur
        time.sleep(2)
        
    except Exception as e:
        print(f"   ❌ Erreur majeure avec cette association : {e}")
        continue

# ==================== SAUVEGARDE FINALE ====================

if all_events:
    # Créer un DataFrame
    df = pd.DataFrame(all_events)
    
    # Enlever les doublons
    df = df.drop_duplicates(subset=['Titre', 'Prix', 'URL_Association'])
    
    # Sauvegarder en Excel
    excel_filename = f"tous_les_evenements_scouts_{time.strftime('%Y%m%d_%H%M%S')}.xlsx"
    
    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        # Feuille principale
        df.to_excel(writer, sheet_name='Tous les événements', index=False)
        
        # Feuille par type
        event_types = df['Type'].unique()
        for event_type in event_types:
            type_df = df[df['Type'] == event_type]
            sheet_name = event_type[:31]
            type_df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    print(f"\n{'='*60}")
    print("🎉 EXTRACTION TERMINÉE AVEC SUCCÈS !")
    print(f"📊 {len(df)} événements récupérés depuis {association_count} associations")
    print(f"💾 Fichier Excel sauvegardé: {excel_filename}")
    
    # Statistiques
    print("\n📈 STATISTIQUES:")
    print("-" * 40)
    type_counts = df['Type'].value_counts()
    for event_type, count in type_counts.items():
        print(f"  {event_type}: {count}")
    
else:
    print("\n❌ Aucun événement n'a été trouvé dans les associations visitées.")

# Fermeture
driver.quit()
print("\n✅ Navigateur fermé. Script terminé !")

if 'excel_filename' in locals() and os.path.exists(excel_filename):
    print(f"\n📁 Le fichier '{excel_filename}' a été créé avec succès.")

🚀 DÉBUT DU SCRAPING COMPLET DES SCOUTS ET GUIDES DE FRANCE
🔍 Début de la collecte des liens d'associations...
✅ 'Tout voir' cliqué.

📄 PAGE 1 - Extraction des associations...
   ✅ 31 nouvelles associations trouvées sur cette page
   📊 Total cumulé : 31 associations
➡️  Passage à la page suivante...

📄 PAGE 2 - Extraction des associations...
   ✅ 30 nouvelles associations trouvées sur cette page
   📊 Total cumulé : 61 associations
➡️  Passage à la page suivante...

📄 PAGE 3 - Extraction des associations...
   ✅ 30 nouvelles associations trouvées sur cette page
   📊 Total cumulé : 91 associations
➡️  Passage à la page suivante...

📄 PAGE 4 - Extraction des associations...
   ✅ 30 nouvelles associations trouvées sur cette page
   📊 Total cumulé : 121 associations
➡️  Passage à la page suivante...

📄 PAGE 5 - Extraction des associations...
   ✅ 30 nouvelles associations trouvées sur cette page
   📊 Total cumulé : 151 associations
➡️  Passage à la page suivante...

📄 PAGE 6 - Extraction des

In [ ]:
SCRAPPING SPAIN

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re
from datetime import datetime

# ==================== CONFIGURATION ====================
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 20)

def scrape_all_scouts_correct():
    """Scrape all scouts CORRECTLY without translation"""
    
    print("🚀 SCRAPING CORRECT - SCOUTS ESPAGNOLS")
    print("=" * 60)
    
    # 1. GO TO PAGE
    print("🔍 Connecting to scout.es...")
    driver.get("https://www.scout.es/grupos-scouts/")
    time.sleep(5)
    
    # Accept cookies
    try:
        cookie_btn = driver.find_element(By.XPATH, "//button[contains(., 'Aceptar')]")
        cookie_btn.click()
        time.sleep(2)
        print("✅ Cookies accepted")
    except:
        print("ℹ️ No cookie popup")
    
    # 2. SELECT "100" TO SHOW ALL
    print("\n🎯 Selecting '100' to show all groups...")
    
    try:
        # Find the select element
        select_element = driver.find_element(
            By.XPATH, 
            "//select[@name='wpgmza_table_1_length']"
        )
        
        # Select "100" option
        select = Select(select_element)
        select.select_by_value("100")
        
        print("✅ '100' selected!")
        time.sleep(10)  # Wait for loading
        
    except Exception as e:
        print(f"⚠️ Could not select 100: {e}")
        time.sleep(5)
    
    # 3. EXTRACT ALL GROUPS
    print("\n🔍 Extracting all scout groups...")
    
    all_groups = []
    
    try:
        # Find all table rows
        rows = driver.find_elements(
            By.XPATH, 
            "//table[@id='wpgmza_table_1']//tr[contains(@class, 'wpgmaps_mlist_row') or @role='row']"
        )
        
        print(f"📊 Found {len(rows)} rows")
        
        # Extract data from each row
        for i, row in enumerate(rows):
            print(f"\n{'='*40}")
            print(f"🎯 GROUP {i+1}/{len(rows)}")
            
            try:
                # Get row HTML for better extraction
                row_html = row.get_attribute('innerHTML')
                
                # Extract REAL name from wpgmza_table_title
                real_name = ""
                name_match = re.search(r'<td[^>]*class="wpgmza_table_title"[^>]*>([^<]+)</td>', row_html)
                if name_match:
                    real_name = name_match.group(1).strip()
                else:
                    # Try with Selenium
                    try:
                        title_cell = row.find_element(By.XPATH, ".//td[@class='wpgmza_table_title']")
                        real_name = title_cell.text.strip()
                    except:
                        cells = row.find_elements(By.TAG_NAME, "td")
                        if cells and len(cells) > 0:
                            real_name = cells[0].text.strip()
                
                if not real_name or real_name == "":
                    real_name = f"Grupo Scout {i+1}"
                
                print(f"📌 NAME: {real_name}")
                
                # Extract address from wpgmza_table_address
                full_address = ""
                address_match = re.search(r'<td[^>]*class="wpgmza_table_address"[^>]*>([^<]+)</td>', row_html)
                if address_match:
                    full_address = address_match.group(1).strip()
                else:
                    try:
                        address_cell = row.find_element(By.XPATH, ".//td[@class='wpgmza_table_address']")
                        full_address = address_cell.text.strip()
                    except:
                        pass
                
                if not full_address:
                    full_address = "Dirección no especificada"
                
                print(f"📍 ADDRESS: {full_address[:60]}..." if len(full_address) > 60 else f"📍 ADDRESS: {full_address}")
                
                # Extract region from wpgmza_table_category
                region = ""
                region_match = re.search(r'<td[^>]*class="(?:wpgmza_table_category|sorting_1)"[^>]*>([^<]+)</td>', row_html)
                if region_match:
                    region = region_match.group(1).strip()
                else:
                    try:
                        region_cell = row.find_element(By.XPATH, ".//td[@class='wpgmza_table_category']")
                        region = region_cell.text.strip()
                    except:
                        pass
                
                if not region:
                    region = "Región no especificada"
                
                print(f"🏙️ REGION: {region}")
                
                # Extract website link
                site_link = ""
                
                # Look in wpgmza_table_link
                link_match = re.search(r'<td[^>]*class="wpgmza_table_link"[^>]*>.*?href="([^"]+)"', row_html)
                if link_match:
                    site_link = link_match.group(1)
                else:
                    # Look for any link in the row
                    try:
                        link_elem = row.find_element(By.TAG_NAME, "a")
                        site_link = link_elem.get_attribute("href")
                    except:
                        # Look for URL in text
                        url_match = re.search(r'(https?://[^\s<>"]+)', row_html)
                        if url_match:
                            site_link = url_match.group(1)
                
                if site_link:
                    print(f"🔗 WEBSITE: {site_link[:60]}...")
                else:
                    print("🔗 WEBSITE: No link found")
                
                # Save group info
                group_info = {
                    'numero': i + 1,
                    'nombre_grupo': real_name,
                    'region': region,
                    'direccion': full_address,
                    'enlace_web': site_link,
                    'estado': 'Con enlace' if site_link else 'Sin enlace'
                }
                
                all_groups.append(group_info)
                
            except Exception as e:
                print(f"❌ Error with group {i+1}: {str(e)[:50]}")
                continue
                
    except Exception as e:
        print(f"❌ Extraction error: {e}")
    
    # 4. SAVE GROUPS LIST
    print("\n" + "=" * 60)
    print("💾 SAVING GROUPS LIST...")
    
    if all_groups:
        df_groups = pd.DataFrame(all_groups)
        
        # Reorder columns
        df_groups = df_groups[['numero', 'nombre_grupo', 'region', 'direccion', 'enlace_web', 'estado']]
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        groups_filename = f"GRUPOS_SCOUTS_ESPANOLES_{timestamp}.xlsx"
        df_groups.to_excel(groups_filename, index=False)
        
        print(f"✅ File saved: {groups_filename}")
        print(f"📊 Total groups: {len(df_groups)}")
        
        # Show first groups
        print("\n📝 FIRST 5 GROUPS:")
        for i, row in df_groups.head(5).iterrows():
            print(f"{row['numero']}. {row['nombre_grupo']}")
            if row['direccion'] != "Dirección no especificada":
                print(f"   📍 {row['direccion'][:50]}...")
            if row['enlace_web']:
                print(f"   🔗 {row['enlace_web'][:40]}...")
            print()
        
        # 5. SCRAPE EVENTS FROM WEBSITES
        print("\n" + "=" * 60)
        print("🌐 SCRAPING EVENTS FROM WEBSITES...")
        
        all_events = []
        
        for i, group in enumerate(all_groups):
            if group['enlace_web'] and group['enlace_web'].startswith('http'):
                print(f"\n▶️ Website {i+1}/{len(all_groups)}: {group['nombre_grupo']}")
                print(f"   🔗 {group['enlace_web'][:50]}...")
                
                events = scrape_events_from_website(
                    group['enlace_web'], 
                    group['nombre_grupo'], 
                    group['direccion']
                )
                
                if events:
                    all_events.extend(events)
                    print(f"   ✅ Found {len(events)} events")
                else:
                    print("   ℹ️ No events found")
                
                # Wait between requests
                if i < len(all_groups) - 1:
                    time.sleep(3)
        
        # 6. SAVE EVENTS
        if all_events:
            df_events = pd.DataFrame(all_events)
            
            # Remove duplicates
            df_events = df_events.drop_duplicates(subset=['grupo', 'evento', 'fecha'])
            
            events_filename = f"EVENTOS_SCOUTS_{timestamp}.xlsx"
            df_events.to_excel(events_filename, index=False)
            
            print(f"\n📊 Total events: {len(df_events)}")
            print(f"💾 Events file: {events_filename}")
            
            # Statistics
            print("\n📈 STATISTICS:")
            print(f"• Groups with events: {df_events['grupo'].nunique()}")
            
            if 'categoria' in df_events.columns:
                print("\n📊 CATEGORIES:")
                cat_counts = df_events['categoria'].value_counts()
                for cat, count in cat_counts.items():
                    print(f"  • {cat}: {count}")
            
            # Show examples
            print("\n📝 EVENT EXAMPLES:")
            for j, (_, row) in enumerate(df_events.head(3).iterrows()):
                event_preview = row['evento'][:80] + "..." if len(row['evento']) > 80 else row['evento']
                print(f"{j+1}. {row['grupo']}")
                print(f"   📅 {row['fecha']}")
                print(f"   📝 {event_preview}")
                print()
        
        else:
            print("\n❌ No events found on websites")
    
    else:
        print("❌ No groups found")
    
    # 7. CLOSE
    driver.quit()
    print("\n" + "=" * 60)
    print("✅ SCRAPING COMPLETED!")

def scrape_events_from_website(url, group_name, group_address):
    """Scrape events from a group website"""
    events = []
    
    try:
        # Save main window
        main_window = driver.current_window_handle
        
        # Open in new tab
        driver.execute_script(f"window.open('{url}');")
        time.sleep(1)
        
        # Switch to new tab
        new_window = [w for w in driver.window_handles if w != main_window][0]
        driver.switch_to.window(new_window)
        
        # Wait for loading
        time.sleep(6)
        
        # Accept cookies if present
        try:
            cookie_selectors = [
                "//button[contains(., 'Aceptar')]",
                "//button[contains(., 'Accept')]",
                "//a[contains(., 'Aceptar')]"
            ]
            
            for selector in cookie_selectors:
                try:
                    btn = driver.find_element(By.XPATH, selector)
                    if btn.is_displayed():
                        btn.click()
                        time.sleep(1)
                        break
                except:
                    continue
        except:
            pass
        
        # SCRAPE EVENTS
        page_html = driver.page_source.lower()
        
        # Event keywords in Spanish
        keywords = [
            'campamento', 'acampada', 'excursión', 'salida',
            'reunión', 'asamblea', 'formación', 'taller',
            'jornada', 'encuentro', 'viaje', 'actividad',
            'evento', 'celebración', 'fiesta', 'convivencia'
        ]
        
        # Search by keywords
        for keyword in keywords:
            if keyword in page_html:
                try:
                    # Find elements with this keyword
                    elements = driver.find_elements(
                        By.XPATH, 
                        f"//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZÁÉÍÓÚÑ', 'abcdefghijklmnopqrstuvwxyzáéíóúñ'), '{keyword}')]"
                    )
                    
                    for element in elements[:5]:  # Max 5 per keyword
                        try:
                            text = element.text.strip()
                            if text and len(text) > 15:
                                # Extract date
                                date_found = extract_date(text)
                                
                                # Categorize
                                category = categorize_event_spanish(text)
                                
                                events.append({
                                    'grupo': group_name,
                                    'direccion_grupo': group_address,
                                    'evento': text[:250],
                                    'fecha': date_found,
                                    'categoria': category,
                                    'fuente': keyword,
                                    'url_sitio': url
                                })
                        except:
                            continue
                except:
                    continue
        
        # Also check page titles
        try:
            headers = driver.find_elements(By.XPATH, "//h1 | //h2 | //h3")
            for header in headers[:10]:
                try:
                    text = header.text.strip()
                    if text and len(text) > 10:
                        text_lower = text.lower()
                        if any(keyword in text_lower for keyword in keywords):
                            date_found = extract_date(text)
                            category = categorize_event_spanish(text)
                            
                            events.append({
                                'grupo': group_name,
                                'direccion_grupo': group_address,
                                'evento': text[:200],
                                'fecha': date_found,
                                'categoria': category,
                                'fuente': 'titulo',
                                'url_sitio': url
                            })
                except:
                    continue
        except:
            pass
        
        # Close tab and return
        driver.close()
        driver.switch_to.window(main_window)
        
    except Exception as e:
        print(f"   ⚠️ Website error: {str(e)[:40]}")
        # Return to main window
        if len(driver.window_handles) > 1:
            try:
                driver.close()
                driver.switch_to.window(main_window)
            except:
                pass
    
    return events

def extract_date(text):
    """Extract date from text"""
    patterns = [
        r'\d{1,2}[-/]\d{1,2}[-/]\d{4}',
        r'\d{1,2}\s+de\s+[a-záéíóúñ]+\s+de\s+\d{4}',
        r'\d{1,2}\s+[a-záéíóúñ]+\s+\d{4}',
        r'\d{4}[-/]\d{1,2}[-/]\d{1,2}'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(0)
    
    return "Fecha no especificada"

def categorize_event_spanish(text):
    """Categorize event in Spanish"""
    text_lower = text.lower()
    
    if any(word in text_lower for word in ['campamento', 'acampada', 'camping']):
        return "Campamento"
    elif any(word in text_lower for word in ['reunión', 'asamblea', 'encuentro']):
        return "Reunión"
    elif any(word in text_lower for word in ['formación', 'curso', 'taller']):
        return "Formación"
    elif any(word in text_lower for word in ['excursión', 'salida', 'viaje']):
        return "Excursión"
    elif any(word in text_lower for word in ['inscripción', 'matrícula']):
        return "Inscripción"
    else:
        return "Otro"

# ==================== RUN ====================
if __name__ == "__main__":
    scrape_all_scouts_correct()

🚀 SCRAPING CORRECT - SCOUTS ESPAGNOLS
🔍 Connecting to scout.es...
ℹ️ No cookie popup

🎯 Selecting '100' to show all groups...
✅ '100' selected!

🔍 Extracting all scout groups...
📊 Found 100 rows

🎯 GROUP 1/100
📌 NAME: Grupo Scout Pyrene 692
📍 ADDRESS: C. Teruel, 14, Barbastro, Huesca, España
🏙️ REGION: Región no especificada
🔗 WEBSITE: https://gs692.scout.es/...

🎯 GROUP 2/100
📌 NAME: Grupo Scout Jaca 219
📍 ADDRESS: C. del Viento, 3, Jaca, Huesca, España
🏙️ REGION: Región no especificada
🔗 WEBSITE: https://219jaca.carrd.co/...

🎯 GROUP 3/100
📌 NAME: Grupo Scout Antorcha 687
📍 ADDRESS: Condesa Mencía 108, (traseras, Parque de la Luz 1 y 2, bajo)...
🏙️ REGION: Región no especificada
🔗 WEBSITE: http://www.antorcha.es/...

🎯 GROUP 4/100
📌 NAME: Grupo Scout Arlanzon 688
📍 ADDRESS: Plaza San Marcelino Champagnat 2, Burgos, España
🏙️ REGION: Región no especificada
🔗 WEBSITE: https://www.scoutsarlanzon.org/...

🎯 GROUP 5/100
📌 NAME: Grupo Scout Ferrol 19
📍 ADDRESS: Estrada Alta do Porto, S/N F